## Feature Engineering - Customer Churn

### By:
Author Name

### Date:
2026-02-23

### Description:

Create a feature engineering pipeline based on the data analysis of the customer churn dataset. 
The feature engineering will be performed using scikit-learn pipelines and transformers to prepare the data for machine learning models.


## 📚 Import libraries

In [ ]:
# base libraries for data science
from pathlib import Path

import pandas as pd
import sklearn as sk
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

## 💾 Load data

In [ ]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

churn_df = pd.read_parquet(DATA_DIR / "02_intermediate/churn_cleaned.parquet", engine="pyarrow")

In [ ]:
# print library versions for reproducibility
print("Pandas version: ", pd.__version__)
print("sklearn version: ", sk.__version__)

In [ ]:
churn_df.info()

In [ ]:
churn_df.head()

## 👷 Data preparation

Select the relevant features for the model and inspect data quality.

In [ ]:
selected_features = [
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
]

churn_features = churn_df[selected_features].copy()
churn_features.info()

### Missing values

In [ ]:
# Replace blank strings ' ' with NaN — common issue in this dataset (TotalCharges)
churn_features = churn_features.replace(r"^\s*$", float("nan"), regex=True)

# TotalCharges is stored as string/category — convert to numeric
churn_features["TotalCharges"] = pd.to_numeric(churn_features["TotalCharges"], errors="coerce")

churn_features.isna().sum()

### Duplicated data

In [ ]:
duplicate_rows = churn_features.duplicated().sum()
print("Number of duplicate rows: ", duplicate_rows)

In [ ]:
churn_features = churn_features.drop_duplicates()
churn_features.info()

## 👨‍🏭 Feature Engineering

Build a scikit-learn preprocessing pipeline to handle imputation, encoding, and scaling for each type of feature.

In [ ]:
# Encode target variable: Yes -> 1, No -> 0
churn_features["Churn"] = (churn_features["Churn"] == "Yes").astype(int)

churn_features["Churn"].value_counts()

In [ ]:
churn_features.sample(5, random_state=42)

In [ ]:
# Numeric features - continuous
cols_numeric = ["tenure", "MonthlyCharges", "TotalCharges"]

# Binary categorical features (Yes/No)
cols_categoric_binary = [
    "Partner",
    "Dependents",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "PaperlessBilling",
]

# Multi-class categorical features
cols_categoric_nominal = ["InternetService", "PaymentMethod"]

# Ordinal categorical features
cols_categoric_ordinal = ["Contract"]  # Month-to-month < One year < Two year

# Integer / binary numeric feature
cols_numeric_binary = ["SeniorCitizen"]  # already encoded as 0/1

In [ ]:
# Numeric pipeline: impute with median + scale
numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Binary categorical pipeline: impute + one-hot encode (drop='if_binary' keeps 1 column)
categorical_binary_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="if_binary", sparse_output=False)),
    ]
)

# Nominal categorical pipeline: impute + one-hot encode
categorical_nominal_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(sparse_output=False)),
    ]
)

# Ordinal categorical pipeline: impute + ordinal encode
contract_order = [["Month-to-month", "One year", "Two year"]]
categorical_ordinal_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(categories=contract_order)),
    ]
)

# Passthrough for already-encoded binary numeric features
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, cols_numeric),
        ("categoric_binary", categorical_binary_pipe, cols_categoric_binary),
        ("categoric_nominal", categorical_nominal_pipe, cols_categoric_nominal),
        ("categoric_ordinal", categorical_ordinal_pipe, cols_categoric_ordinal),
        ("passthrough", "passthrough", cols_numeric_binary),
    ]
)

preprocessor

**Pipeline description:**

1. **Numeric Pipeline** — `["tenure", "MonthlyCharges", "TotalCharges"]`:
   - `SimpleImputer(strategy="median")`: fills missing values with the median.
   - `StandardScaler()`: standardizes features to zero mean and unit variance.

2. **Binary Categorical Pipeline** — e.g. `Partner`, `Dependents`, `OnlineSecurity`, …:
   - `SimpleImputer(strategy="most_frequent")`: fills missing values with the mode.
   - `OneHotEncoder(drop="if_binary")`: encodes Yes/No as a single 0/1 column.

3. **Nominal Categorical Pipeline** — `["InternetService", "PaymentMethod"]`:
   - `SimpleImputer(strategy="most_frequent")`: fills missing values with the mode.
   - `OneHotEncoder()`: creates one binary column per category.

4. **Ordinal Categorical Pipeline** — `["Contract"]`:
   - `SimpleImputer(strategy="most_frequent")`: fills missing values with the mode.
   - `OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]])`: encodes with meaningful order (0 < 1 < 2).

5. **Passthrough** — `["SeniorCitizen"]`: already encoded as 0/1, passed through unchanged.

## Example of the data preprocessing pipeline

### Train / Test split

In [ ]:
X_features = churn_features.drop("Churn", axis="columns")
Y_target = churn_features["Churn"]

# 80% train, 20% test — stratify to preserve churn ratio
x_train, x_test, y_train, y_test = train_test_split(
    X_features, Y_target, test_size=0.2, random_state=42, stratify=Y_target
)

print(f"Train set: {x_train.shape}, Test set: {x_test.shape}")

### Preprocessing pipeline

In [ ]:
# Fit preprocessor on training data only to avoid data leakage
preprocessor.fit(x_train)

feature_names = preprocessor.get_feature_names_out()

x_train_transformed = pd.DataFrame(preprocessor.transform(x_train), columns=feature_names)
x_test_transformed = pd.DataFrame(preprocessor.transform(x_test), columns=feature_names)

x_train_transformed.info()

In [ ]:
x_train_transformed.head()

In [ ]:
# Verify no missing values remain after transformation
print("Missing values in transformed train set:", x_train_transformed.isna().sum().sum())
print("Missing values in transformed test set: ", x_test_transformed.isna().sum().sum())

## 📊 Analysis of Results and Conclusions

The feature engineering pipeline successfully transforms the raw churn dataset into a format ready for machine learning:

- **Numeric features** (`tenure`, `MonthlyCharges`, `TotalCharges`) are imputed and standardized, ensuring algorithms sensitive to scale (e.g., logistic regression, SVM) are not biased.
- **Binary Yes/No features** are efficiently encoded into single 0/1 columns, keeping dimensionality low.
- **Nominal features** (`InternetService`, `PaymentMethod`) are one-hot encoded to avoid introducing false ordinal relationships.
- **Ordinal feature** (`Contract`) is encoded respecting the natural order: Month-to-month (0) < One year (1) < Two year (2).
- **No data leakage**: the preprocessor is fitted only on training data and then applied to the test set.

The final transformed dataset has no missing values and all features are numeric, making it directly usable for classification models.


## 💡 Proposals and Ideas

1. **Advanced imputation**: Consider `KNNImputer` or `IterativeImputer` for `TotalCharges` if missing values are non-random (MNAR/MAR).

2. **New features via feature engineering**:
   - `services_count`: count of active services per customer (streaming, security, backup, etc.) to capture total engagement.
   - `charges_per_month_ratio`: `TotalCharges / tenure` as a proxy for pricing stability.
   - `is_new_customer`: binary flag for `tenure < 6` months, since new customers may have higher churn risk.

3. **Feature scaling alternatives**: Try `MinMaxScaler` or `RobustScaler` for numeric features if outliers are present in charges.

4. **Target encoding**: For high-cardinality categorical features (e.g., `PaymentMethod`), compare one-hot vs. target encoding impact on model performance.

5. **Class imbalance**: Explore SMOTE or class weighting in the model, since churn datasets are typically imbalanced (~26% churn in this dataset).


## 📖 References

- https://joserzapata.github.io/post/ciencia-datos-proyecto-python/4-feat_eng/
- https://joserzapata.github.io/courses/python-ciencia-datos/ml/
- Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow — Aurélien Géron
- Scikit-learn documentation: https://scikit-learn.org/stable/modules/preprocessing.html
